## Convert a .grd to GeoTIFF
- `.grd` files produced by gmt sometimes have weirdness in the longitude
- The most basic is converting, say -90 to 270 but some other strange stuff happens at times
- Sometimes, here it will say that the longitude is, say 30 deg, and when you load into GIS it's still 30 deg, but off by 360 deg. In this case, we just have to add (or maybe subtract in some cases?) 360 deg until it magically work. There has to be a better way but I haven't found one yet. 

In [2]:
# ! cd /Volumes/T9/2025-01-07_tibet/S1/F1/intf/2025004_2025016/
# ! ls
# ! proj_ra2ll.csh /Volumes/T9/2025-01-07_tibet/S1/F1/intf/2025004_2025016/trans.dat /Volumes/T9/2025-01-07_tibet/S1/F1/intf/2025004_2025016/yphase.grd /Volumes/T9/2025-01-07_tibet/S1/F1/intf/2025004_2025016/yphase_ll.grd

In [1]:

import xarray as xr
import rioxarray
import os

merge_dir = '/Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/'
filetypes = ["corr_ll", "los_ll", "los_ll_dtr", "xphase_mask_ll", "yphase_mask_ll", "phasefilt_ll","phasefilt_mask_ll", "xphase_ll","yphase_ll"]    # "los_ll", "xphase_mask_ll", "yphase_mask_ll", "phasefilt_ll", "imagfilt_ll", "realfilt_ll"


## Parse metadata from the grd file and write to a text file
# Find the .h5 files in the `intf` level directory and pull metadata
# date, track, direction


for file in filetypes:
    file_path = merge_dir + file + '.grd'
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue

    ds = xr.open_dataset(file_path)

    # --- Normalize longitude to -180..180 ---
    if "lon" in ds.coords:
        lon = ds.lon.values
        # Wrap into [-180, 180]
        lon_wrapped = ((lon + 180) % 360) - 180
        ds = ds.assign_coords(lon=lon_wrapped)

    ds = ds.rename({"lon": "x", "lat": "y"})

    ds.rio.write_crs("EPSG:4326", inplace=True)  # WGS 84
    out_file = os.path.join(merge_dir, file + '.tiff')
    ds.rio.to_raster(out_file)

    print(f"Wrote {out_file}")


Wrote /Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/corr_ll.tiff
File not found: /Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/los_ll.grd
File not found: /Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/los_ll_dtr.grd
Wrote /Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/xphase_mask_ll.tiff
Wrote /Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/yphase_mask_ll.tiff
Wrote /Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/phasefilt_ll.tiff
Wrote /Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/phasefilt_mask_ll.tiff
File not found: /Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/xphase_ll.grd
File not found: /Volumes/T9_InSAR/2026-10-09_panama/NISAR/D040/RSLC/intf/2026274_2026286/yphase_ll.grd


## Manually correct a strange grid


In [2]:
dir = '/Volumes/T9/2025-07-30_kamchatka/S1/A111/stitched_frames/F1/SLC/'
file = "S1_20250804_071617_F1_ll"

ds = xr.open_dataset(dir + file + '.grd')

if "lon" in ds.coords:
    lon = ds.lon.values
    # Wrap into [-180, 180]
    lon_wrapped = ((lon + 180) % 360) - 180
    ds = ds.assign_coords(lon=lon_wrapped)




ds = ds.rename({"lon": "x", "lat": "y"})
for var in ds.data_vars:
    ds[var].data = ds[var].data[::-1, ::-1]  # flip rows and columns

# for var in ds.data_vars:
#     ds[var].data = ds[var].data[::-1, ::]  # flip only rows


ds.rio.write_crs("EPSG:4326", inplace=True)  # WGS 84
out_file = os.path.join(merge_dir, file + '.tiff')
ds.rio.to_raster(out_file)


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/hyin/miniforge3/envs/gmtsar/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3549, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/x7/q8y3dbkj0gv831q8bnwtym_c0000gp/T/ipykernel_86291/2835208053.py", line 4, in <module>
    ds = xr.open_dataset(dir + file + '.grd')
  File "/Users/hyin/miniforge3/envs/gmtsar/lib/python3.13/site-packages/xarray/backends/api.py", line 668, in open_dataset
    engine = plugins.guess_engine(filename_or_obj)
  File "/Users/hyin/miniforge3/envs/gmtsar/lib/python3.13/site-packages/xarray/backends/plugins.py", line 194, in guess_engine
    raise ValueError(error_msg)
ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'scipy', 'rasterio']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO depe